In [1]:
# Installing our toolboxes (the ! tells Jupyter to install them on your PC)
!pip install pandas requests sqlalchemy pymysql

In [2]:
import pandas as pd
import requests

# 1. The URL for FEMA's public flood insurance claims database (v3)
claims_url = "https://www.fema.gov/api/open/v3/NfipClaims"

# 2. Tell FEMA exactly which columns and rows we want (Florida claims from 2020+)
claims_params = {
    "$filter": "state eq 'FL' and yearOfLoss ge 2020",
    "$select": (
        "id,dateOfLoss,yearOfLoss,state,countyCode,reportedZipCode,"
        "ratedFloodZone,occupancyType,buildingDamageAmount,"
        "netBuildingPaymentAmount,netContentsPaymentAmount,"
        "totalBuildingInsuranceCoverage,buildingDeductibleCode,"
        "elevatedBuildingIndicator,elevationCertificateIndicator,"
        "eventDesignationNumber"
    ),
    "$top": 10000,   # Pull 10,000 rows so it downloads fast
    "$format": "json"
}

# 3. Send the request to FEMA and convert the response into a Pandas table (DataFrame)
print("Downloading claims data from FEMA...")
response = requests.get(claims_url, params=claims_params, timeout=60)
data_json = response.json()

# Convert the JSON list into a neat rows-and-columns DataFrame
df_claims = pd.DataFrame(data_json["NfipClaims"])

print(f"Success! Downloaded {len(df_claims)} rows and {len(df_claims.columns)} columns.")

C:\Users\korad\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Success! Downloaded 10000 rows and 16 columns.


In [3]:
# 1. The URL for FEMA's public flood insurance policies database (v3)
policies_url = "https://www.fema.gov/api/open/v3/NfipPolicies"

# 2. Ask for 10,000 Florida policies from 2022 onward (NO $select so it never fails on column names)
policies_params = {
    "$filter": "propertyState eq 'FL' and policyEffectiveDate ge '2022-01-01'",
    "$top": 10000,
    "$format": "json"
}

print("Downloading policies data from FEMA...")
pol_response = requests.get(policies_url, params=policies_params, timeout=60)

if pol_response.status_code != 200:
    print("Server Error Details:", pol_response.text)
else:
    pol_json = pol_response.json()
    df_policies = pd.DataFrame(pol_json["NfipPolicies"])
    print(f"Success! Downloaded {len(df_policies)} policy rows and {len(df_policies.columns)} columns.")

Success! Downloaded 10000 policy rows and 87 columns.


In [4]:
# 1. Save raw backups to your folder so we never have to re-download them
df_claims.to_csv("raw_claims.csv", index=False)
df_policies.to_csv("raw_policies.csv", index=False)
print("Saved raw_claims.csv and raw_policies.csv to your folder!\n")

# 2. See the exact column names FEMA gave us in df_policies
print("--- POLICIES COLUMNS ---")
print(list(df_policies.columns))

# 3. Preview the first 5 rows of claims
df_claims.head(5)

Saved raw_claims.csv and raw_policies.csv to your folder!

--- POLICIES COLUMNS ---
['agricultureStructureIndicator', 'asOfDate', 'baseFloodElevation', 'basementEnclosureCrawlspaceType', 'cancellationDateOfFloodPolicy', 'condominiumCoverageTypeCode', 'construction', 'crsClassCode', 'buildingDeductibleCode', 'contentsDeductibleCode', 'elevatedBuildingIndicator', 'elevationCertificateIndicator', 'elevationDifference', 'federalPolicyFee', 'ratedFloodZone', 'hfiaaSurcharge', 'houseOfWorshipIndicator', 'locationOfContents', 'lowestAdjacentGrade', 'lowestFloorElevation', 'nonProfitIndicator', 'numberOfFloorsInInsuredBuilding', 'obstructionType', 'occupancyType', 'originalConstructionDate', 'originalNBDate', 'policyCost', 'policyCount', 'policyEffectiveDate', 'policyTerminationDate', 'policyTermIndicator', 'postFIRMConstructionIndicator', 'primaryResidenceIndicator', 'rateMethod', 'regularEmergencyProgramIndicator', 'smallBusinessIndicatorBuilding', 'totalBuildingInsuranceCoverage', 'totalCon

,id,dateOfLoss,yearOfLoss,state,countyCode,reportedZipCode,ratedFloodZone,occupancyType,buildingDamageAmount,netBuildingPaymentAmount,netContentsPaymentAmount,totalBuildingInsuranceCoverage,buildingDeductibleCode,elevatedBuildingIndicator,elevationCertificateIndicator,eventDesignationNumber
0,5448679,2024-09-26T00:00:00.000Z,2024,FL,12103,33708,AE,18,571913.0,500000.00,0.00,500000,E,False,NaN,FL0124
1,5448693,2022-09-28T00:00:00.000Z,2022,FL,12071,33908,AE,15,245077.0,243827.07,0.00,500000,F,True,3,FL0222
2,5448739,2024-09-26T00:00:00.000Z,2024,FL,12071,33903,AE,11,91985.0,89984.55,0.00,206000,2,False,NaN,FL0124
3,5448758,2024-09-26T00:00:00.000Z,2024,FL,12103,33704,AE,11,201100.0,211607.44,682.11,250000,2,False,NaN,FL0124
4,5448797,2022-09-28T00:00:00.000Z,2022,FL,12071,33957,AE,11,22509.0,20509.23,0.00,250000,2,True,NaN,FL0222


In [5]:
# 1. Save raw backups to your folder so we never have to re-download them
df_claims.to_csv("raw_claims.csv", index=False)
df_policies.to_csv("raw_policies.csv", index=False)
print("Saved raw_claims.csv and raw_policies.csv to your folder!\n")

# 2. Inspect column data types and missing values
print("--- CLAIMS TABLE STRUCTURE ---")
df_claims.info()

# 3. Preview the first 5 rows of claims
df_claims.head(5)

Saved raw_claims.csv and raw_policies.csv to your folder!

--- CLAIMS TABLE STRUCTURE ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              10000 non-null  int64  
 1   dateOfLoss                      10000 non-null  str    
 2   yearOfLoss                      10000 non-null  int64  
 3   state                           10000 non-null  str    
 4   countyCode                      9997 non-null   str    
 5   reportedZipCode                 10000 non-null  str    
 6   ratedFloodZone                  10000 non-null  str    
 7   occupancyType                   10000 non-null  int64  
 8   buildingDamageAmount            8841 non-null   float64
 9   netBuildingPaymentAmount        10000 non-null  float64
 10  netContentsPaymentAmount        10000 non-null  float64
 11  totalBuildingIn

,id,dateOfLoss,yearOfLoss,state,countyCode,reportedZipCode,ratedFloodZone,occupancyType,buildingDamageAmount,netBuildingPaymentAmount,netContentsPaymentAmount,totalBuildingInsuranceCoverage,buildingDeductibleCode,elevatedBuildingIndicator,elevationCertificateIndicator,eventDesignationNumber
0,5448679,2024-09-26T00:00:00.000Z,2024,FL,12103,33708,AE,18,571913.0,500000.00,0.00,500000,E,False,NaN,FL0124
1,5448693,2022-09-28T00:00:00.000Z,2022,FL,12071,33908,AE,15,245077.0,243827.07,0.00,500000,F,True,3,FL0222
2,5448739,2024-09-26T00:00:00.000Z,2024,FL,12071,33903,AE,11,91985.0,89984.55,0.00,206000,2,False,NaN,FL0124
3,5448758,2024-09-26T00:00:00.000Z,2024,FL,12103,33704,AE,11,201100.0,211607.44,682.11,250000,2,False,NaN,FL0124
4,5448797,2022-09-28T00:00:00.000Z,2022,FL,12071,33957,AE,11,22509.0,20509.23,0.00,250000,2,True,NaN,FL0222


In [6]:
claims_clean = df_claims.copy()

In [7]:
import pandas as pd
claims_clean["dateOfLoss"] = pd.to_datetime(claims_clean["dateOfLoss"], errors="coerce").dt.date

In [8]:
claims_clean.head()

,id,dateOfLoss,yearOfLoss,state,countyCode,reportedZipCode,ratedFloodZone,occupancyType,buildingDamageAmount,netBuildingPaymentAmount,netContentsPaymentAmount,totalBuildingInsuranceCoverage,buildingDeductibleCode,elevatedBuildingIndicator,elevationCertificateIndicator,eventDesignationNumber
0,5448679,2024-09-26,2024,FL,12103,33708,AE,18,571913.0,500000.00,0.00,500000,E,False,NaN,FL0124
1,5448693,2022-09-28,2022,FL,12071,33908,AE,15,245077.0,243827.07,0.00,500000,F,True,3,FL0222
2,5448739,2024-09-26,2024,FL,12071,33903,AE,11,91985.0,89984.55,0.00,206000,2,False,NaN,FL0124
3,5448758,2024-09-26,2024,FL,12103,33704,AE,11,201100.0,211607.44,682.11,250000,2,False,NaN,FL0124
4,5448797,2022-09-28,2022,FL,12071,33957,AE,11,22509.0,20509.23,0.00,250000,2,True,NaN,FL0222


In [9]:
money_columns = [
    "buildingDamageAmount",
    "netBuildingPaymentAmount",
    "netContentsPaymentAmount",
    "totalBuildingInsuranceCoverage"

]

for col in money_columns:
    claims_clean[col] = pd.to_numeric(claims_clean[col], errors="coerce").fillna(0)

In [10]:
claims_clean["total_claim_paid_usd"] = (claims_clean["netBuildingPaymentAmount"] +
                                        claims_clean["netContentsPaymentAmount"])

In [11]:
claims_clean["eventDesignationNumber"] = (
    claims_clean["eventDesignationNumber"].fillna("ATTRITIONAL").replace("", "ATTRITIONAL"))

In [12]:
claims_clean[["dateOfLoss", "eventDesignationNumber", "buildingDamageAmount", "total_claim_paid_usd"]].head()

,dateOfLoss,eventDesignationNumber,buildingDamageAmount,total_claim_paid_usd
0,2024-09-26,FL0124,571913.0,500000.00
1,2022-09-28,FL0222,245077.0,243827.07
2,2024-09-26,FL0124,91985.0,89984.55
3,2024-09-26,FL0124,201100.0,212289.55
4,2022-09-28,FL0222,22509.0,20509.23


In [13]:
# 1. Official FEMA Lookup Table: Code -> Dollar Deductible
deductible_lookup = {
    "0": 500,   "1": 1000,  "2": 2000,  "3": 3000,
    "4": 4000,  "5": 5000,  "9": 750,   "A": 10000,
    "B": 15000, "C": 20000, "D": 25000, "E": 50000,
    "F": 1250,  "G": 1500,  "H": 200
}

In [14]:
claims_clean["building_deductible_usd"] = (
    claims_clean["buildingDeductibleCode"].astype(str).map(deductible_lookup).fillna(1000)
)

In [15]:
claims_clean["risk_zone_tier"] = claims_clean["ratedFloodZone"].apply(
    lambda zone: "High Risk (A/V Zone)" if zone.startswith(("A", "V")) else "Moderate/Low Risk (X Zone)"
)

In [16]:
claims_clean["zip_code"] = claims_clean["reportedZipCode"].str[:5]

In [17]:
claims_clean["elevationCertificateIndicator"].value_counts(dropna=False)

elevationCertificateIndicator
NaN    8840
3       696
1       253
2       178
4        32
B         1
Name: count, dtype: int64

In [18]:
claims_clean["elevationCertificateIndicator"] = (
    claims_clean["elevationCertificateIndicator"]
    .fillna("No Certificate (Unverified)")
    .replace({
        "1": "Has Certificate (Verified)",
        "2": "Has Certificate (Verified)",
        "3": "Has Certificate (Verified)",
        "4": "Has Certificate (Verified)",
        "B": "No Certificate (Unverified)",
        1: "Has Certificate (Verified)",
        2: "Has Certificate (Verified)",
        3: "Has Certificate (Verified)",
        4: "Has Certificate (Verified)"
    })
)

In [19]:
claims_clean["elevationCertificateIndicator"].value_counts(dropna=False)

elevationCertificateIndicator
No Certificate (Unverified)    8841
Has Certificate (Verified)     1159
Name: count, dtype: int64

In [20]:
claims_clean["elevatedBuildingIndicator"] = (
    claims_clean["elevatedBuildingIndicator"]
    .fillna("Unknown")
    .replace({True: "Elevated", False: "Non-Elevated", "Y": "Elevated", "N": "Non-Elevated"})
)

In [21]:
claims_clean["elevatedBuildingIndicator"].value_counts(dropna=False)

elevatedBuildingIndicator
Non-Elevated    8326
Elevated        1674
Name: count, dtype: int64

In [22]:
print("Decoded Deductibles and Risk Tiers:")
claims_clean[["ratedFloodZone", "risk_zone_tier", "buildingDeductibleCode", "building_deductible_usd", "elevationCertificateIndicator", "elevatedBuildingIndicator"]].head(5)

Decoded Deductibles and Risk Tiers:


,ratedFloodZone,risk_zone_tier,buildingDeductibleCode,building_deductible_usd,elevationCertificateIndicator,elevatedBuildingIndicator
0,AE,High Risk (A/V Zone),E,50000.0,No Certificate (Unverified),Non-Elevated
1,AE,High Risk (A/V Zone),F,1250.0,Has Certificate (Verified),Elevated
2,AE,High Risk (A/V Zone),2,2000.0,No Certificate (Unverified),Non-Elevated
3,AE,High Risk (A/V Zone),2,2000.0,No Certificate (Unverified),Non-Elevated
4,AE,High Risk (A/V Zone),2,2000.0,No Certificate (Unverified),Elevated


In [23]:
claims_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              10000 non-null  int64  
 1   dateOfLoss                      10000 non-null  object 
 2   yearOfLoss                      10000 non-null  int64  
 3   state                           10000 non-null  str    
 4   countyCode                      9997 non-null   str    
 5   reportedZipCode                 10000 non-null  str    
 6   ratedFloodZone                  10000 non-null  str    
 7   occupancyType                   10000 non-null  int64  
 8   buildingDamageAmount            10000 non-null  float64
 9   netBuildingPaymentAmount        10000 non-null  float64
 10  netContentsPaymentAmount        10000 non-null  float64
 11  totalBuildingInsuranceCoverage  10000 non-null  int64  
 12  buildingDeductibleCode          9702 non-nul

# --- CELL 7: CLEAN & STANDARDIZE THE POLICIES TABLE ---

In [24]:
policies_clean = df_policies.copy()

In [25]:
policies_clean.head()


,agricultureStructureIndicator,asOfDate,baseFloodElevation,basementEnclosureCrawlspaceType,cancellationDateOfFloodPolicy,condominiumCoverageTypeCode,construction,crsClassCode,buildingDeductibleCode,contentsDeductibleCode,...,latitude,longitude,buildingOnFederalLand,buildingPurpose,seasonallyOccupied,fullRiskPremium,buildingOverWaterType,foundationType,preFIRMConstructionIndicator,id
0,False,2026-09-08T00:00:00.000Z,NaN,0.0,NaN,A,False,6.0,D,NaN,...,27.4,-82.7,False,R,False,4252.0,1.0,1,False,205821924
1,False,2026-09-08T00:00:00.000Z,8.0,0.0,NaN,N,False,3.0,F,F,...,25.8,-80.4,False,R,False,862.0,1.0,1,False,205822157
2,False,2026-09-08T00:00:00.000Z,10.0,NaN,NaN,N,False,5.0,F,NaN,...,27.9,-82.4,False,R,False,3639.0,1.0,1,False,205825014
3,False,2026-09-08T00:00:00.000Z,7.0,0.0,NaN,N,False,3.0,F,F,...,25.9,-80.3,False,R,False,319.0,1.0,1,False,205828256
4,False,2026-09-08T00:00:00.000Z,8.0,0.0,NaN,N,False,5.0,D,D,...,26.1,-81.8,False,N,False,5720.0,1.0,1,False,205832831


In [26]:
# 1. Convert policyEffectiveDate into a real date and extract the Policy Year
policies_clean["policyEffectiveDate"] = pd.to_datetime(
    policies_clean["policyEffectiveDate"], errors="coerce"
)
policies_clean["policy_year"] = policies_clean["policyEffectiveDate"].dt.year.fillna(2022).astype(int)
policies_clean["policyEffectiveDate"] = policies_clean["policyEffectiveDate"].dt.date

In [27]:
# 2. Standardize State and ZIP code to match claims_clean
policies_clean["state"] = policies_clean["propertyState"].fillna("FL")
policies_clean["zip_code"] = policies_clean["reportedZipCode"].fillna("00000").astype(str).str[:5]

In [28]:
# 3. Decode Deductible Code and Flood Risk Zone using the exact same rules from Codespace[13] where asigned as "deductible_lookup"
policies_clean["building_deductible_usd"] = (
    policies_clean["buildingDeductibleCode"].astype(str).map(deductible_lookup).fillna(1000)
)

In [29]:
policies_clean["ratedFloodZone"] = policies_clean["ratedFloodZone"].fillna("UNKNOWN").astype(str)
policies_clean["risk_zone_tier"] = policies_clean["ratedFloodZone"].apply(
    lambda zone: "High Risk (A/V Zone)" if zone.startswith(("A", "V")) else "Moderate/Low Risk (X Zone)"
)

In [30]:
# 4. Clean Premium, Coverage, and Policy Count numbers
# (FEMA stores premium in totalInsurancePremiumOfThePolicy or policyCost)
premium_col = (
    "totalInsurancePremiumOfThePolicy"
    if "totalInsurancePremiumOfThePolicy" in policies_clean.columns
    else "policyCost"
)
policies_clean["written_premium_usd"] = pd.to_numeric(policies_clean[premium_col], errors="coerce").fillna(0)
policies_clean["building_coverage_usd"] = pd.to_numeric(
    policies_clean["totalBuildingInsuranceCoverage"], errors="coerce"
).fillna(0)
policies_clean["policy_count"] = pd.to_numeric(policies_clean["policyCount"], errors="coerce").fillna(1)

print("Cell 7 complete! Preview of cleaned policies:")
policies_clean[["policyEffectiveDate", "zip_code", "risk_zone_tier", "written_premium_usd", "building_coverage_usd"]].head(5)

Cell 7 complete! Preview of cleaned policies:


,policyEffectiveDate,zip_code,risk_zone_tier,written_premium_usd,building_coverage_usd
0,2022-01-01,34228,High Risk (A/V Zone),1393,250000
1,2022-01-01,33174,High Risk (A/V Zone),334,209000
2,2022-01-01,33602,High Risk (A/V Zone),472,250000
3,2022-01-01,33015,High Risk (A/V Zone),484,145000
4,2022-01-01,34112,High Risk (A/V Zone),2254,339700


In [36]:
# --- CELL 8: KEEP ONLY CLEAN COLUMNS & SAVE FOR MYSQL ---

# 1. Select the final columns for our Claims table
final_claims_cols = [
    "id", "dateOfLoss", "yearOfLoss", "state", "zip_code",
    "ratedFloodZone", "risk_zone_tier", "occupancyType",
    "elevatedBuildingIndicator", "elevationCertificateIndicator",
    "eventDesignationNumber", "building_deductible_usd",
    "totalBuildingInsuranceCoverage", "buildingDamageAmount",
    "netBuildingPaymentAmount", "netContentsPaymentAmount", "total_claim_paid_usd"
]
claims_final = claims_clean[final_claims_cols]

# 2. Select the final columns for our Policies table
final_policies_cols = [
    "id", "policyEffectiveDate", "policy_year", "state", "zip_code",
    "ratedFloodZone", "risk_zone_tier", "occupancyType",
    "elevatedBuildingIndicator", "building_deductible_usd",
    "policy_count", "written_premium_usd", "building_coverage_usd"
]
policies_final = policies_clean[final_policies_cols]

# 3. Save both clean tables to your project folder
claims_final.to_csv("clean_claims.csv", index=False)
policies_final.to_csv("clean_policies.csv", index=False)
print("✅ Saved clean_claims.csv and clean_policies.csv!\n")

# 4. YOUR FIRST BUSINESS INSIGHT: Compare Claims Paid by Risk Zone Tier
summary = claims_final.groupby("risk_zone_tier").agg(
    total_claims=("id", "count"),
    avg_payout_usd=("total_claim_paid_usd", "mean"),
    total_payout_usd=("total_claim_paid_usd", "sum")
).round()

summary.style.format({
    "avg_payout_usd": "{:,.0f}",
    "total_payout_usd": "{:,.0f}"
})


✅ Saved clean_claims.csv and clean_policies.csv!



,total_claims,avg_payout_usd,total_payout_usd
risk_zone_tier,,,
High Risk (A/V Zone),9252,"94,383","873,228,973"
Moderate/Low Risk (X Zone),748,"44,501","33,287,113"


# Linking jupyternotebook to mySQL

In [42]:
# --- CELL 11: SEND CLEAN TABLES DIRECTLY INTO MYSQL ---
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# 1. Enter your MySQL login details
mysql_user = "root"
mysql_password = quote_plus("My_password")
mysql_host = "localhost"
mysql_port = "3306"
database_name = "reinsurance_db"

# 2. Build the connection bridge to MySQL
connection_url = f"mysql+pymysql://{mysql_user}:{mysql_password}@{mysql_host}:{mysql_port}/{database_name}"
engine = create_engine(connection_url)

# 3. Push both clean DataFrames into MySQL as tables
print("Uploading fact_claims to MySQL...")
claims_final.to_sql(name="fact_claims", con=engine, if_exists="replace", index=False)

print("Uploading fact_policies to MySQL...")
policies_final.to_sql(name="fact_policies", con=engine, if_exists="replace", index=False)

print("✅ Success! Both tables are now inside MySQL (reinsurance_db)!")

Uploading fact_claims to MySQL...
Uploading fact_policies to MySQL...
✅ Success! Both tables are now inside MySQL (reinsurance_db)!
